# IGDB 002: IGDB field review

**Per-column EDA on pipeline IGDB artifacts**

Profiles all IGDB game columns in `igdb_games.parquet`. Field docs: [`field_docs.py`](../../src/steam_review_ml/igdb/field_docs.py).


# Executive Summary

**Question:**  
For each IGDB game column in our joined parquet, what is the **value shape**, **typical content**, and **likely v2 utility**?

**Result:**  
_Update after running the per-field section._

**Recommendation / Decision:**  
_Tag which fields to wire for V2a vs V2b vs defer._


# Data Sources

**Primary:** `artifacts/igdb/igdb_games.parquet` (from `recs_job_igdb_games.py`)

**Join context:** `artifacts/igdb/igdb_join_report.json` (optional)

**Pipeline refresh:** `python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json`


# Notebook Roadmap

1. Setup + load parquet
2. Overview table (all IGDB fields)
3. Per-field profiles
4. Key Findings (manual)


# Analysis


## Setup


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.igdb.constants import IGDB_GAMES_PARQUET, STEAM_JOIN_COLS, V2_CORE_FIELDS
from steam_review_ml.igdb.field_docs import GAME_FIELD_DOCS, V2_FIELD_NOTES

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
IGDB_DIR = REPO_ROOT / "artifacts/igdb"
PARQUET_PATH = IGDB_DIR / IGDB_GAMES_PARQUET
REPORT_PATH = IGDB_DIR / "igdb_join_report.json"

SAMPLE_APP_IDS = [753420, 646910, 512900]

print(f"REPO_ROOT={REPO_ROOT}")


REPO_ROOT=/home/ryanr/workspace/steam_recommendations


In [2]:
if not PARQUET_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {PARQUET_PATH}. Run:\n"
        "  python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json"
    )

df = pd.read_parquet(PARQUET_PATH)
join_report = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.is_file() else {}

IGDB_GAME_FIELDS = sorted(c for c in df.columns if c not in STEAM_JOIN_COLS and c != "igdb_name")
absent_from_parquet = sorted(set(GAME_FIELD_DOCS) - set(IGDB_GAME_FIELDS) - {"name"})

print(f"rows={len(df)} cols={len(df.columns)}")
print(f"IGDB_GAME_FIELDS ({len(IGDB_GAME_FIELDS)}): {IGDB_GAME_FIELDS}")
print(f"catalog match rate: {join_report.get('match_rate', 'n/a')}")
if absent_from_parquet:
    print(f"Fields in GAME_FIELD_DOCS but not in parquet: {absent_from_parquet}")


rows=314 cols=59
IGDB_GAME_FIELDS (54): ['age_ratings', 'aggregated_rating', 'aggregated_rating_count', 'alternative_names', 'artworks', 'bundles', 'checksum', 'collections', 'cover', 'created_at', 'dlcs', 'expanded_games', 'expansions', 'external_games', 'first_release_date', 'franchise', 'franchises', 'game_engines', 'game_localizations', 'game_modes', 'game_status', 'game_type', 'genres', 'hypes', 'involved_companies', 'keywords', 'language_supports', 'multiplayer_modes', 'parent_game', 'platforms', 'player_perspectives', 'ports', 'rating', 'rating_count', 'release_dates', 'remakes', 'remasters', 'screenshots', 'similar_games', 'slug', 'standalone_expansions', 'status', 'storyline', 'summary', 'tags', 'themes', 'total_rating', 'total_rating_count', 'updated_at', 'url', 'version_parent', 'version_title', 'videos', 'websites']
catalog match rate: 0.9968253968253968
Fields in GAME_FIELD_DOCS but not in parquet: ['category', 'collection', 'follows', 'forks']


In [3]:
def igdb_field_doc(field: str) -> dict[str, str]:
    doc = GAME_FIELD_DOCS.get(field, {})
    return {
        "igdb_type": doc.get("type", ""),
        "igdb_description": doc.get("description", ""),
        "deprecated": doc.get("deprecated", False),
        "v2_note": V2_FIELD_NOTES.get(field, ""),
    }


def field_populated(value: Any) -> bool:
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except (TypeError, ValueError):
        pass
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, (list, dict, set, tuple, np.ndarray)):
        return len(value) > 0
    return True


def _as_list(value: Any) -> list[Any]:
    if value is None:
        return []
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple, set)):
        return list(value)
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass
    return [value]


def value_kind(value: Any) -> str:
    if not field_populated(value):
        return "empty"
    if isinstance(value, str):
        return "string"
    if isinstance(value, (int, np.integer)):
        return "integer"
    if isinstance(value, (float, np.floating)):
        return "float"
    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return value_kind(value.item())
        return "id_list" if np.issubdtype(value.dtype, np.integer) else "array"
    if isinstance(value, list):
        return "id_list" if value and all(isinstance(x, (int, np.integer)) for x in value) else "list"
    return type(value).__name__


def profile_column(series: pd.Series) -> dict[str, Any]:
    populated = series.map(field_populated)
    populated_s = series.loc[populated]
    kinds = populated_s.map(value_kind)
    kind_mode = kinds.mode().iloc[0] if len(kinds) else "empty"

    doc = igdb_field_doc(str(series.name))
    out: dict[str, Any] = {
        "field": series.name,
        "dtype": str(series.dtype),
        "coverage_pct": round(float(populated.mean()) * 100, 2),
        "value_kind": kind_mode,
        "v2_core": series.name in V2_CORE_FIELDS,
        "igdb_type": doc["igdb_type"],
        "igdb_description": doc["igdb_description"],
        "v2_note": doc["v2_note"],
    }

    if kind_mode == "string":
        lens = populated_s.astype(str).str.len()
        out["len_median"] = int(lens.median())
        out["len_p95"] = int(lens.quantile(0.95))
    elif kind_mode in {"integer", "float"}:
        nums = pd.to_numeric(populated_s, errors="coerce").dropna()
        if len(nums):
            out["min"] = float(nums.min())
            out["median"] = float(nums.median())
            out["max"] = float(nums.max())
    elif kind_mode == "id_list":
        lengths = populated_s.map(lambda v: len(_as_list(v)))
        out["list_len_median"] = float(lengths.median())
        out["list_len_p95"] = float(lengths.quantile(0.95))
        out["unique_ids"] = int(pd.Series([x for v in populated_s for x in _as_list(v)]).nunique())

    return out


def format_sample(value: Any, *, max_len: int = 240) -> str:
    if not field_populated(value):
        return "<empty>"
    if isinstance(value, str):
        text = value.replace("\n", " ")
        return text if len(text) <= max_len else text[: max_len - 3] + "..."
    if isinstance(value, np.ndarray):
        return f"array({value.tolist()})"
    return repr(value)[:max_len]


def display_field_profile(data: pd.DataFrame, field: str) -> None:
    series = data[field]
    prof = profile_column(series)
    doc = igdb_field_doc(field)
    v2_tag = " **V2_CORE**" if prof.get("v2_core") else ""
    dep_tag = " *(deprecated)*" if doc.get("deprecated") else ""
    display(Markdown(f"### `{field}`{v2_tag}{dep_tag}"))
    if doc.get("igdb_type"):
        display(Markdown(f"**IGDB type:** `{doc['igdb_type']}`"))
    if doc.get("igdb_description"):
        display(Markdown(f"**IGDB description:** {doc['igdb_description']}"))
    if doc.get("v2_note"):
        display(Markdown(f"**V2 note:** {doc['v2_note']}"))

    summary_rows = {
        k: v
        for k, v in prof.items()
        if k not in {"field", "igdb_type", "igdb_description", "v2_note", "v2_core"}
    }
    display(pd.DataFrame([summary_rows]))

    sample_df = (
        data.loc[data["app_id"].isin(SAMPLE_APP_IDS), ["app_id", "app_name", field]]
        .sort_values("app_id")
        .assign(sample=lambda d: d[field].map(format_sample))
        .drop(columns=[field])
    )
    display(sample_df)

    extra = data.loc[series.map(field_populated), field].head(3)
    print("Additional samples:")
    for app_id, val in zip(data.loc[extra.index, "app_id"], extra):
        print(f"  app_id={app_id}: {format_sample(val)}")


## Overview


In [4]:
overview = pd.DataFrame([profile_column(df[c]) for c in IGDB_GAME_FIELDS])
overview = overview.sort_values(["v2_core", "coverage_pct"], ascending=[False, False]).reset_index(drop=True)
display(overview)

display(overview.loc[overview["v2_core"], ["field", "coverage_pct", "value_kind", "igdb_type", "v2_note"]])


,field,dtype,coverage_pct,value_kind,v2_core,igdb_type,igdb_description,v2_note,list_len_median,list_len_p95,unique_ids,min,median,max,len_median,len_p95
0,game_modes,object,100.00,id_list,True,Array of Game Mode IDs,Modes of gameplay,V2a — Jaccard on game mode ID sets.,2.0,4.0,6.0,NaN,NaN,NaN,NaN,NaN
1,genres,object,100.00,id_list,True,Array of Genre IDs,Genres of the game,V2a — Jaccard on genre ID sets (entity lookup ...,3.0,5.0,20.0,NaN,NaN,NaN,NaN,NaN
2,summary,str,100.00,string,True,String,A description of the game,"V2b — USE dot(query_review, summary). Ready wi...",NaN,NaN,NaN,NaN,NaN,NaN,356.0,903.0
3,themes,object,98.41,id_list,True,Array of Theme IDs,Themes of the game,V2a — Jaccard on theme ID sets.,2.0,5.0,22.0,NaN,NaN,NaN,NaN,NaN
4,player_perspectives,object,96.18,id_list,True,Array of Player Perspective IDs,The main perspective of the player,V2a — Jaccard on perspective ID sets.,1.0,2.0,7.0,NaN,NaN,NaN,NaN,NaN
5,keywords,object,91.40,id_list,True,Array of Keyword IDs,Associated keywords,V2a — Jaccard on keyword ID sets; ~91% coverag...,12.0,84.0,1707.0,NaN,NaN,NaN,NaN,NaN
6,franchises,object,29.94,id_list,True,Array of Franchise IDs,Other franchises the game belongs to,V2a — Jaccard on franchise ID sets; sparse (~3...,1.0,2.0,91.0,NaN,NaN,NaN,NaN,NaN
7,checksum,str,100.00,string,False,uuid,Hash of the object,Audit / reproducibility only.,NaN,NaN,NaN,NaN,NaN,NaN,36.0,36.0
8,cover,int64,100.00,integer,False,Reference ID for Cover,The cover of this game,Media reference — not for text ranker.,NaN,NaN,NaN,2.884000e+03,1.149230e+05,5.701310e+05,NaN,NaN
9,created_at,int64,100.00,integer,False,datetime,Date this was initially added to the IGDB data...,IGDB record metadata.,NaN,NaN,NaN,1.297956e+09,1.467616e+09,1.675468e+09,NaN,NaN


,field,coverage_pct,value_kind,igdb_type,v2_note
0,game_modes,100.00,id_list,Array of Game Mode IDs,V2a — Jaccard on game mode ID sets.
1,genres,100.00,id_list,Array of Genre IDs,V2a — Jaccard on genre ID sets (entity lookup ...
2,summary,100.00,string,String,"V2b — USE dot(query_review, summary). Ready wi..."
3,themes,98.41,id_list,Array of Theme IDs,V2a — Jaccard on theme ID sets.
4,player_perspectives,96.18,id_list,Array of Player Perspective IDs,V2a — Jaccard on perspective ID sets.
5,keywords,91.40,id_list,Array of Keyword IDs,V2a — Jaccard on keyword ID sets; ~91% coverag...
6,franchises,29.94,id_list,Array of Franchise IDs,V2a — Jaccard on franchise ID sets; sparse (~3...


## Per-field review

One subsection per IGDB game column.


In [5]:
for field in IGDB_GAME_FIELDS:
    display_field_profile(df, field)
    print()


### `age_ratings`

**IGDB type:** `Array of Age Rating IDs`

**IGDB description:** The PEGI rating

,dtype,coverage_pct,value_kind,list_len_median,list_len_p95,unique_ids
0,object,85.35,id_list,5.0,7.0,1327


,app_id,app_name,sample
2,512900,Streets of Rogue,"array([125538, 195813, 39086, 181520, 57827, 1..."
1,646910,The Crew 2,"array([111035, 111034, 91834, 92244, 215725, 2..."
0,753420,Dungreed,"array([119531, 55417, 98238, 187890])"


...

**IGDB description:** Websites associated with this game

**V2 note:** Media / link reference.

,dtype,coverage_pct,value_kind,list_len_median,list_len_p95,unique_ids
0,object,100.0,id_list,11.0,17.0,3388


,app_id,app_name,sample
2,512900,Streets of Rogue,"array([831925, 761475, 761476, 110813, 16428, ..."
1,646910,The Crew 2,"array([63721, 312584, 833260, 833259, 833261, ..."
0,753420,Dungreed,"array([920873, 920874, 756045, 756041, 520086,..."


Additional samples:
  app_id=753420: array([920873, 920874, 756045, 756041, 520086, 128835, 63833, 128836, 340645, 520085])
  app_id=646910: array([63721, 312584, 833260, 833259, 833261, 833262, 63719, 63722, 312583, 340578, 39649, 63723, 63720])
  app_id=512900: array([831925, 761475, 761476, 110813, 16428, 121120, 236719, 338319, 655211, 16429, 16427])

